# KonkaniVani ASR - Complete Retraining with Periodic Testing

## Overview
- **Data**: 84 hours from KonkaniRawSpeechCorpus + existing data
- **Testing**: Every 5 epochs to monitor progress
- **Expected**: Working model by epoch 20-30
- **Time**: ~8-12 hours on Kaggle GPU

## Setup
1. Upload your data as Kaggle dataset
2. Enable GPU accelerator
3. Run all cells
4. Monitor progress in output

## 1. Setup Environment

In [ ]:
# Check GPU
!nvidia-smi

import torch
print(f"\nPyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# Install dependencies
!pip install -q soundfile librosa torchaudio tensorboard tqdm

## 2. Upload Data to Kaggle

### Option A: Upload as Kaggle Dataset (Recommended)
1. Zip your data locally:
   ```bash
   zip -r konkani_complete_data.zip \
     KonkaniRawSpeechCorpus/ \
     data/konkani-asr-v0/ \
     data/vocab.json
   ```
2. Upload to Kaggle Datasets
3. Add dataset to this notebook

### Option B: Use Kaggle API
Upload from your machine using Kaggle API

In [ ]:
# Check if data is available
import os
from pathlib import Path

# Adjust these paths based on your Kaggle dataset
DATA_ROOT = Path('/kaggle/input/konkani-complete-data')  # Change to your dataset name

print("Checking data availability...")
print(f"Data root exists: {DATA_ROOT.exists()}")

if DATA_ROOT.exists():
    print(f"\nContents:")
    !ls -lh {DATA_ROOT}
else:
    print("\n⚠️  Data not found! Please add your dataset to this notebook.")

## 3. Copy Project Files to Working Directory

In [ ]:
# Create working directory structure
!mkdir -p /kaggle/working/models
!mkdir -p /kaggle/working/data/audio_processing
!mkdir -p /kaggle/working/checkpoints
!mkdir -p /kaggle/working/logs
!mkdir -p /kaggle/working/outputs

print("✓ Directory structure created")

## 4. Prepare Data - Create Manifests

In [ ]:
%%writefile /kaggle/working/prepare_data.py
#!/usr/bin/env python3
"""
Prepare KonkaniRawSpeechCorpus data for training
"""
import json
from pathlib import Path
import soundfile as sf
from tqdm import tqdm
import random

def parse_transcript_file(txt_path):
    """Parse the transcript .txt file"""
    with open(txt_path, 'r', encoding='utf-8') as f:
        content = f.read()
    
    if 'RECORDED TEXT ::' in content:
        parts = content.split('RECORDED TEXT ::')
        if len(parts) > 1:
            text_part = parts[1].split('TEXT TRANSLITERATION ::')[0]
            text = text_part.strip()
            return text
    return None

def create_manifests(corpus_dir, existing_dir, output_dir):
    """Create combined manifests"""
    corpus_path = Path(corpus_dir)
    output_path = Path(output_dir)
    output_path.mkdir(parents=True, exist_ok=True)
    
    print("="*70)
    print("PREPARING DATA")
    print("="*70)
    
    # Process raw corpus
    print("\n1. Processing KonkaniRawSpeechCorpus...")
    wav_files = list(corpus_path.rglob('*.wav'))
    print(f"   Found {len(wav_files):,} audio files")
    
    samples = []
    skipped = 0
    
    for wav_path in tqdm(wav_files, desc="Processing"):
        txt_path = wav_path.with_suffix('.txt')
        if not txt_path.exists():
            skipped += 1
            continue
        
        text = parse_transcript_file(txt_path)
        if not text or len(text.strip()) == 0:
            skipped += 1
            continue
        
        try:
            info = sf.info(wav_path)
            if info.duration < 0.5 or info.duration > 15.0:
                skipped += 1
                continue
            
            samples.append({
                'audio_filepath': str(wav_path.absolute()),
                'text': text,
                'duration': info.duration,
                'language': 'knn_Deva'
            })
        except:
            skipped += 1
    
    print(f"   ✓ Processed {len(samples):,} samples")
    print(f"   ✗ Skipped {skipped:,} samples")
    
    # Load existing data
    print("\n2. Loading existing data...")
    existing_path = Path(existing_dir)
    for split in ['train', 'val', 'test']:
        manifest_file = existing_path / f'{split}.json'
        if manifest_file.exists():
            with open(manifest_file, 'r') as f:
                existing_samples = [json.loads(line) for line in f]
            samples.extend(existing_samples)
            print(f"   Added {len(existing_samples):,} from {split}.json")
    
    # Shuffle and split
    print("\n3. Creating splits...")
    random.shuffle(samples)
    
    n_train = int(len(samples) * 0.8)
    n_val = int(len(samples) * 0.1)
    
    splits = {
        'train': samples[:n_train],
        'val': samples[n_train:n_train+n_val],
        'test': samples[n_train+n_val:]
    }
    
    # Save manifests
    for split_name, split_samples in splits.items():
        manifest_path = output_path / f'{split_name}.json'
        with open(manifest_path, 'w', encoding='utf-8') as f:
            for sample in split_samples:
                f.write(json.dumps(sample, ensure_ascii=False) + '\n')
        
        total_hours = sum(s['duration'] for s in split_samples) / 3600
        print(f"   {split_name:5s}: {len(split_samples):6,} samples ({total_hours:5.1f}h)")
    
    print(f"\n✓ Manifests saved to: {output_path}")
    return output_path

if __name__ == '__main__':
    manifest_dir = create_manifests(
        corpus_dir='/kaggle/input/konkani-complete-data/KonkaniRawSpeechCorpus/Data',
        existing_dir='/kaggle/input/konkani-complete-data/data/konkani-asr-v0/splits/manifests',
        output_dir='/kaggle/working/data/manifests'
    )
    print("\n✓ Data preparation complete!")

In [ ]:
# Run data preparation
!python /kaggle/working/prepare_data.py

## 5. Copy Model Files

In [ ]:
# Copy your model architecture, audio processor, tokenizer, etc.
# You'll need to upload these as part of your dataset or paste them here

print("Copy your project files:")
print("  - models/konkanivani_asr.py")
print("  - data/audio_processing/audio_processor.py")
print("  - data/audio_processing/text_tokenizer.py")
print("  - data/audio_processing/dataset.py")
print("  - data/vocab.json")
print("\nThese should be in your Kaggle dataset.")

## 6. Training Script with Periodic Testing

In [ ]:
%%writefile /kaggle/working/train_with_testing.py
#!/usr/bin/env python3
"""
Training with periodic transcription testing
"""
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from tqdm import tqdm
import json
from pathlib import Path
import sys
from datetime import datetime

# Import your modules (adjust paths as needed)
sys.path.insert(0, '/kaggle/working')

# Training configuration
CONFIG = {
    'train_manifest': '/kaggle/working/data/manifests/train.json',
    'val_manifest': '/kaggle/working/data/manifests/val.json',
    'test_manifest': '/kaggle/working/data/manifests/test.json',
    'vocab_path': '/kaggle/input/konkani-complete-data/data/vocab.json',
    
    'batch_size': 16,
    'num_epochs': 100,
    'learning_rate': 3e-4,
    'weight_decay': 1e-4,
    'grad_clip': 5.0,
    
    'ctc_weight': 0.8,  # FIXED: was 0.3
    'attn_weight': 0.2,
    
    'test_every_n_epochs': 5,
    'num_test_samples': 5,
    
    'checkpoint_dir': '/kaggle/working/checkpoints',
    'save_every_n_epochs': 5,
}

def test_transcription_quality(model, tokenizer, test_samples, device, epoch):
    """Test actual transcription quality"""
    from data.audio_processing.audio_processor import AudioProcessor
    
    model.eval()
    processor = AudioProcessor(sample_rate=16000, n_mels=80)
    
    results = {'epoch': epoch, 'samples': []}
    blank_probs = []
    unique_counts = []
    
    with torch.no_grad():
        for sample in test_samples:
            try:
                waveform = processor.load_audio(sample['audio_filepath'])
                features = processor.compute_features(waveform, apply_augment=False)
                features_batch = features.unsqueeze(0).to(device)
                
                encoder_out, _ = model.encoder(features_batch)
                ctc_logits = model.ctc_head(encoder_out)
                probs = torch.softmax(ctc_logits, dim=-1)
                preds = torch.argmax(probs, dim=-1)
                
                blank_prob = probs[0, :, tokenizer.blank_id].mean().item()
                unique_tokens = len(torch.unique(preds[0]))
                
                # Decode
                pred_tokens = preds[0].cpu().numpy()
                transcription = decode_ctc(pred_tokens, tokenizer)
                
                blank_probs.append(blank_prob)
                unique_counts.append(unique_tokens)
                
                results['samples'].append({
                    'ground_truth': sample['text'][:80],
                    'prediction': transcription[:80] if transcription else '(empty)',
                    'blank_prob': blank_prob,
                    'unique_tokens': unique_tokens
                })
            except Exception as e:
                results['samples'].append({'error': str(e)})
    
    results['avg_blank_prob'] = sum(blank_probs) / len(blank_probs) if blank_probs else 1.0
    results['avg_unique_tokens'] = sum(unique_counts) / len(unique_counts) if unique_counts else 0
    results['is_working'] = results['avg_blank_prob'] < 0.85 and results['avg_unique_tokens'] > 10
    
    return results

def decode_ctc(tokens, tokenizer):
    """CTC decoding"""
    decoded = []
    prev_token = None
    special_tokens = {'<pad>', '<blank>', '<sos>', '<eos>', '<unk>'}
    
    for token in tokens:
        if token == tokenizer.blank_id:
            prev_token = None
            continue
        if token == prev_token:
            continue
        if token in tokenizer.idx_to_char:
            char = tokenizer.idx_to_char[token]
            if char not in special_tokens:
                decoded.append(char)
        prev_token = token
    
    return ''.join(decoded)

def print_test_results(results):
    """Print test results"""
    print("\n" + "="*80)
    print(f"TRANSCRIPTION TEST - EPOCH {results['epoch']}")
    print("="*80)
    print(f"\nMetrics:")
    print(f"  Blank prob: {results['avg_blank_prob']:.1%}")
    print(f"  Unique tokens: {results['avg_unique_tokens']:.1f}")
    print(f"  Status: {'✅ WORKING!' if results['is_working'] else '❌ Not yet'}")
    
    print(f"\nSamples:")
    for i, s in enumerate(results['samples'][:3], 1):
        if 'error' not in s:
            print(f"\n  [{i}] GT: {s['ground_truth']}")
            print(f"      PR: {s['prediction']}")
            print(f"      Blank: {s['blank_prob']:.1%}, Tokens: {s['unique_tokens']}")

# Main training loop would go here
# (Full implementation in actual notebook)

if __name__ == '__main__':
    print("Training script loaded. Run training cells to start.")

## 7. Start Training

In [ ]:
# Initialize training
# (You'll need to implement full training loop or import from your training script)

print("Starting training with periodic testing...")
print(f"Testing every {CONFIG['test_every_n_epochs']} epochs")
print(f"Total epochs: {CONFIG['num_epochs']}")
print(f"\nExpected time: ~8-12 hours on Kaggle GPU")

# Your training loop here

## 8. Monitor Progress

In [ ]:
# View test results
import json
from pathlib import Path

test_results_dir = Path('/kaggle/working/checkpoints')
test_files = sorted(test_results_dir.glob('test_results_epoch_*.json'))

print("Test Results Summary:")
print("="*70)
print(f"{'Epoch':>6} {'Blank%':>8} {'Tokens':>8} {'Status':>12}")
print("-"*70)

for test_file in test_files:
    with open(test_file, 'r') as f:
        results = json.load(f)
    
    epoch = results['epoch']
    blank = results['avg_blank_prob'] * 100
    tokens = results['avg_unique_tokens']
    status = '✅ Working' if results['is_working'] else '❌ Not yet'
    
    print(f"{epoch:>6} {blank:>7.1f}% {tokens:>7.1f} {status:>12}")

## 9. Download Best Checkpoint

In [ ]:
# Download best model
from IPython.display import FileLink

best_checkpoint = '/kaggle/working/checkpoints/best_model.pt'
if Path(best_checkpoint).exists():
    print("✓ Best checkpoint ready for download")
    FileLink(best_checkpoint)
else:
    print("Checkpoint not found yet. Training in progress...")